(fluid_property_backends_label)=

# Comparison of fluid property back ends

TESPy supports multiple fluid property libraries through its wrapper system:

| Engine | Wrapper | Installation | Notes |
|:-------|:--------|:-------------|:------|
| [CoolProp](https://coolprop.org) | `CoolPropWrapper` | installed with TESPy | default engine, reference quality multiparameter equations of state |
| [REFPROP](https://www.nist.gov/srd/refprop) (via CoolProp) | `CoolPropWrapper` | separate license required | highest accuracy and robustness, also for mixtures |
| [thermopack](https://github.com/thermotools/thermopack) | `ThermopackWrapper` | `uv add thermopack` or `pip install thermopack` | cubic and SAFT type equations of state, fast and robust mixture flashes |
| [iapws](https://github.com/jjgomera/iapws/) | `IAPWSWrapper` | `pip install iapws` | water only, untested |
| [pyromat](https://github.com/chmarti1/PYroMat/) | `PyromatWrapper` | `pip install pyromat` | untested |
| custom data | `IncompressibleFluidWrapper` | installed with TESPy | fits functions to tabular liquid data |

This notebook compares CoolProp and thermopack in two different contexts:

- a complete model of an ORC plant with a **pure working fluid** and a heat
  pump with a **zeotropic mixture**
- performance benchmarks of the individual property calls

The deviations in the results between the engines result from the differences
between the underlying equations of state. For for accuracy the multiparameter
formulations (CoolProp HEOS, REFPROP) are the reference.

In [1]:
import time

import pandas as pd

from tespy.components import CycleCloser
from tespy.components import Compressor
from tespy.components import Pump
from tespy.components import SimpleHeatExchanger
from tespy.components import Turbine
from tespy.components import Valve
from tespy.connections import Connection
from tespy.networks import Network
from tespy.tools.fluid_properties.wrappers import CoolPropWrapper
from tespy.tools.fluid_properties.wrappers import ThermopackWrapper


REPEATS = 3


def make_network():
    nw = Network(iterinfo=False)
    nw.units.set_defaults(
        temperature="degC", pressure="bar", pressure_difference="bar",
        power="kW", heat="kW"
    )
    return nw


def run_system(builder, fluid, engine):
    times = []
    for _ in range(REPEATS):
        nw, metrics = builder(fluid, engine)
        start = time.perf_counter()
        nw.solve("design")
        times.append(time.perf_counter() - start)
        nw.assert_convergence()
    return {"solve time / s": min(times), **metrics()}


def compare(builder, engines):
    results = {}
    for label, fluid, engine in engines:
        try:
            results[label] = run_system(builder, fluid, engine)
        except Exception as e:
            print(f"{label}: failed - {str(e)[:120]}")
    df = pd.DataFrame(results).T
    reference = df.index[0]
    deviation = (
        df.drop("solve time / s", axis=1)
        .div(df.drop("solve time / s", axis=1).loc[reference])
        .sub(1).abs().max(axis=1)
    )
    df["max deviation vs reference"] = deviation.map("{:.2%}".format)
    return df

import CoolProp.CoolProp as CP

try:
    CP.AbstractState("REFPROP", "Propane")
    REFPROP_AVAILABLE = True
except ValueError:
    print("REFPROP is not available on this machine, skipping it.")
    REFPROP_AVAILABLE = False


## Pure working fluids

In this section we use an Organic Rankine Cycle with n-pentane as working 
fluid. The evaporation temperature is at 120 °C, condensation at 30 °C. We
compare the CoolProp HEOS reference with the REFPROP back end and the cubic
(PR, SRK) and PC-SAFT equations of state from thermopack.

In [2]:
def build_orc(fluid, engine):
    nw = make_network()
    cc = CycleCloser("cycle closer")
    pu = Pump("feed pump")
    ev = SimpleHeatExchanger("evaporator")
    tu = Turbine("turbine")
    cd = SimpleHeatExchanger("condenser")

    c1 = Connection(cc, "out1", pu, "in1", label="1")
    c2 = Connection(pu, "out1", ev, "in1", label="2")
    c3 = Connection(ev, "out1", tu, "in1", label="3")
    c4 = Connection(tu, "out1", cd, "in1", label="4")
    c5 = Connection(cd, "out1", cc, "in1", label="5")
    nw.add_conns(c1, c2, c3, c4, c5)

    name = fluid.split("::")[-1]
    c1.set_attr(fluid={fluid: 1}, fluid_engines={name: engine}, m=10)
    c3.set_attr(T=120, x=1)
    c5.set_attr(T=30, x=0)
    pu.set_attr(eta_s=0.75)
    tu.set_attr(eta_s=0.85)
    ev.set_attr(pr=1)
    cd.set_attr(pr=1)

    def metrics():
        return {
            "eta thermal": (abs(tu.P.val) - pu.P.val) / ev.Q.val,
            "P turbine / kW": tu.P.val,
            "p evaporation / bar": c3.p.val,
            "p condensation / bar": c5.p.val,
        }

    return nw, metrics


compare(build_orc, [
    ("CoolProp HEOS", "n-Pentane", CoolPropWrapper),
] + ([
    ("CoolProp REFPROP", "REFPROP::n-Pentane", CoolPropWrapper),
] if REFPROP_AVAILABLE else []) + [
    ("thermopack PR", "PR::NC5", ThermopackWrapper),
    ("thermopack SRK", "SRK::NC5", ThermopackWrapper),
    ("thermopack PC-SAFT", "PC-SAFT::NC5", ThermopackWrapper),
])

,solve time / s,eta thermal,P turbine / kW,p evaporation / bar,p condensation / bar,max deviation vs reference
CoolProp HEOS,0.018050,0.150297,-773.623705,9.074375,0.820049,0.00%
CoolProp REFPROP,0.021044,0.150297,-773.623705,9.074375,0.820049,0.00%
thermopack PR,0.021195,0.150130,-779.491426,9.114723,0.822627,0.76%
thermopack SRK,0.020732,0.149817,-788.938563,9.237444,0.816356,1.98%
thermopack PC-SAFT,0.048279,0.149730,-784.567748,9.031892,0.820647,1.41%


The computational performance in terms of speed is similar for all engines
because the solver overhead of TESPy dominates the individual property calls in
such a simple model. HEOS and REFPROP give identical results here, since both
implement the same reference formulation. The cubic equations of state deviate
by up to two percent from the reference in the pressure levels, the resulting
thermal efficiency is nearly identical.

## Zeotropic mixtures

For mixtures the CoolProp HEOS mixture back end is rather unstable and its
flash routines are slow (as of version `8.0`). The REFPROP back end is accurate
and robust, but it requires a separate license. The thermopack flash routines
are quite fast and robust, and no additional license is required.

The test case for the mixture is a simple heat pump working with a zeotropic
50/50 (molar) mixture of propane and isobutane.

In [3]:
def build_heat_pump(fluid, engine):
    nw = make_network()
    cc = CycleCloser("cycle closer")
    cp = Compressor("compressor")
    cd = SimpleHeatExchanger("condenser")
    va = Valve("valve")
    ev = SimpleHeatExchanger("evaporator")

    c1 = Connection(cc, "out1", cp, "in1", label="1")
    c2 = Connection(cp, "out1", cd, "in1", label="2")
    c3 = Connection(cd, "out1", va, "in1", label="3")
    c4 = Connection(va, "out1", ev, "in1", label="4")
    c5 = Connection(ev, "out1", cc, "in1", label="5")
    nw.add_conns(c1, c2, c3, c4, c5)

    name = fluid.split("::")[-1]
    c1.set_attr(fluid={fluid: 1}, fluid_engines={name: engine}, m=1)
    c3.set_attr(T=40, x=0)
    c5.set_attr(T=5, x=1)
    cp.set_attr(eta_s=0.8)
    cd.set_attr(pr=1)
    ev.set_attr(pr=1)

    def metrics():
        return {
            "COP": abs(cd.Q.val) / cp.P.val,
            "P compressor / kW": cp.P.val,
            "p evaporation / bar": c5.p.val,
            "p condensation / bar": c3.p.val,
        }

    return nw, metrics


cp_fluid = "Propane[0.5]&Isobutane[0.5]|molar"
tp_fluid = "C3[0.5]&IC4[0.5]|molar"

compare(build_heat_pump, ([
    ("CoolProp HEOS", f"HEOS::{cp_fluid}", CoolPropWrapper),
] + [
    ("CoolProp REFPROP", f"REFPROP::{cp_fluid}", CoolPropWrapper),
] if REFPROP_AVAILABLE else []) + [
    ("thermopack PR", f"PR::{tp_fluid}", ThermopackWrapper),
    ("thermopack SRK", f"SRK::{tp_fluid}", ThermopackWrapper),
])

,solve time / s,COP,P compressor / kW,p evaporation / bar,p condensation / bar,max deviation vs reference
CoolProp HEOS,1.358514,5.300033,63.124962,2.811957,9.175032,0.00%
CoolProp REFPROP,0.024407,5.300042,63.124715,2.811956,9.175028,0.00%
thermopack PR,0.025642,5.317745,62.943829,2.797596,9.112522,0.68%
thermopack SRK,0.026910,5.293879,63.919183,2.791947,9.179236,1.26%


All results agree within about one percent, however, the solve times differ 
significantly for the HEOS back end mixture, which is nearly 100 times slower
than all the other ones. On top of that, HEOS mixture routines often fail
entirely. E.g. for a mixture of methane and ethane the wrapper cannot be
constructed, while REFPROP and thermopack handle these successfully:

In [4]:
try:
    CoolPropWrapper("Methane[0.9]&Ethane[0.1]|molar")
except ValueError as e:
    print(f"CoolProp HEOS: {e}")

wrapper = ThermopackWrapper("C1[0.9]&C2[0.1]|molar", "PR")
print(
    "thermopack PR: T_bubble(20 bar) ="
    f" {wrapper.T_bubble(20e5) - 273.15:.1f} °C, T_dew(20 bar) ="
    f" {wrapper.T_dew(20e5) - 273.15:.1f} °C"
)

CoolProp HEOS: critical point finding routine found 8 critical points
thermopack PR: T_bubble(20 bar) = -103.8 °C, T_dew(20 bar) = -75.4 °C


How to model and analyse systems with zeotropic mixtures is showcased in a
{ref}`separate notebook <zeotropic_heat_pump_label>`.

## Individual property calls

To identify the origin of the performance differences, we can time the
individual wrapper calls.

In [5]:
def microbenchmark(wrapper, n):
    import numpy as np

    p_range = np.linspace(wrapper._p_crit * 0.2, wrapper._p_crit * 0.8, n)
    h_range = np.array([
        wrapper.h_pQ(p, 0) * 0.5 + wrapper.h_pQ(p, 1) * 0.5 for p in p_range
    ])
    T_range = np.linspace(
        wrapper.T_sat(p_range[0]) + 20, wrapper._T_crit * 1.2, n
    )
    try:
        s_range = np.array([
            wrapper.s_ph(p, h) for p, h in zip(p_range, h_range)
        ])
    except Exception:
        s_range = None

    operations = {
        "T_ph (two-phase)": (wrapper.T_ph, zip(p_range, h_range)),
        "d_ph (two-phase)": (wrapper.d_ph, zip(p_range, h_range)),
        "s_ph (two-phase)": (wrapper.s_ph, zip(p_range, h_range)),
        "T_ps (two-phase)": (
            wrapper.T_ps,
            None if s_range is None else zip(p_range, s_range)
        ),
        "h_pT (gas)": (wrapper.h_pT, zip(p_range, T_range)),
        "T_sat": (wrapper.T_sat, ((p,) for p in p_range)),
        "h_pQ (dew)": (wrapper.h_pQ, ((p, 1) for p in p_range)),
    }
    results = {}
    for key, (method, args) in operations.items():
        if args is None:
            results[key] = "failed: s_ph setup"
            continue
        try:
            start = time.perf_counter()
            for arg in args:
                method(*arg)
            results[key] = f"{(time.perf_counter() - start) / n * 1e6:.1f}"
        except Exception as e:
            results[key] = "failed: " + str(e)[:30]
    return results


def compare_calls(engines, n):
    results = {
        label: microbenchmark(engine(fluid, back_end), n)
        for label, fluid, back_end, engine in engines
    }
    return pd.DataFrame(results).T.round(1)

In [6]:
compare_calls([
    ("CoolProp HEOS", "CO2", None, CoolPropWrapper),
] + ([
    ("CoolProp REFPROP", "CO2", "REFPROP", CoolPropWrapper),
] if REFPROP_AVAILABLE else []) + [
    ("thermopack PR", "CO2", "PR", ThermopackWrapper),
    ("thermopack SRK", "CO2", "SRK", ThermopackWrapper),
    ("thermopack PC-SAFT", "CO2", "PC-SAFT", ThermopackWrapper),
], n=200)

,T_ph (two-phase),d_ph (two-phase),s_ph (two-phase),T_ps (two-phase),h_pT (gas),T_sat,h_pQ (dew)
CoolProp HEOS,4.0,4.0,4.3,4.0,11.4,0.5,2.2
CoolProp REFPROP,20.4,22.8,21.1,21.1,11.1,13.8,13.7
thermopack PR,24.6,35.6,37.3,26.4,20.8,22.3,26.7
thermopack SRK,23.9,34.9,36.3,24.1,20.4,19.9,26.2
thermopack PC-SAFT,332.2,384.1,404.1,341.1,212.5,287.2,357.1


In [7]:
compare_calls([
    (
        "CoolProp HEOS",
        "Propane[0.5]&Isobutane[0.5]|molar",
        None,
        CoolPropWrapper
    ),
] + ([
    (
        "CoolProp REFPROP",
        "Propane[0.5]&Isobutane[0.5]|molar",
        "REFPROP",
        CoolPropWrapper
    ),
] if REFPROP_AVAILABLE else []) + [
    ("thermopack PR", "C3[0.5]&IC4[0.5]|molar", "PR", ThermopackWrapper),
    ("thermopack SRK", "C3[0.5]&IC4[0.5]|molar", "SRK", ThermopackWrapper),
], n=20)

,T_ph (two-phase),d_ph (two-phase),s_ph (two-phase),T_ps (two-phase),h_pT (gas),T_sat,h_pQ (dew)
CoolProp HEOS,88276.4,86428.4,79378.9,97385.5,5737.9,173.0,158.9
CoolProp REFPROP,632.7,640.7,621.8,619.9,82.5,63.3,62.7
thermopack PR,67.0,65.7,63.3,473.8,29.1,30.6,31.7
thermopack SRK,47.0,59.6,62.9,444.4,29.1,28.2,31.8
